In [13]:
from kafka.admin import KafkaAdminClient, NewTopic

admin = KafkaAdminClient(
    bootstrap_servers="kafka:9092",
)

In [14]:
topic = NewTopic(
    name="weather_stream",
    num_partitions=1,
    replication_factor=1
)

In [19]:
if "weather_stream" not in admin.list_topics():
    admin.create_topics([topic])

In [16]:
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers="kafka:9092",
    value_serializer=lambda message: json.dumps(message).encode("utf-8")
)

In [17]:
message = {"msg": "Hello, Kafka"}
producer.send("weather_stream", value=message)
producer.flush()

In [ ]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    "weather_stream",
    bootstrap_servers="kafka:9092",
    auto_offset_reset="earliest",
    value_deserializer=lambda message: json.loads(message.decode("utf-8"))
)

for message in consumer:
    print(f"Received message: {message.value}")

Received message: {'msg': 'Hello, Kafka'}
Received message: {'msg': 'Hello, Kafka'}
Received message: {'latitude': 48.853382, 'longitude': 2.355208, 'generationtime_ms': 0.07534027099609375, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 36.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'wind_speed_10m': 'm/s', 'weather_code': 'wmo code'}, 'current': {'time': '2026-11-10T13:45', 'interval': 900, 'temperature_2m': 12.3, 'wind_speed_10m': 1.91, 'weather_code': 80}, 'city': 'Paris', 'country': 'France'}
Received message: {'latitude': 48.853382, 'longitude': 2.355208, 'generationtime_ms': 0.07534027099609375, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 36.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'wind_speed_10m': 'm/s', 'weather_code': 'wmo code'}, 'current': {'time': '2026-11-10T13:45', 'interval': 900, 'temperature_2m

KeyboardInterrupt: 

In [21]:
cities = [
    {
        "city": "Paris",
        "country": "France",
        "latitude": 48.8566,
        "longitude": 2.3522
    },
    {
        "city": "Seville",
        "country": "Spain",
        "latitude": 37.3891,
        "longitude": -5.9845
    },
    {
        "city": "Dubai",
        "country": "United Arab Emirates",
        "latitude": 25.2048,
        "longitude": 55.2708
    },
    {
        "city": "Reykjavik",
        "country": "Iceland",
        "latitude": 64.1466,
        "longitude": -21.9426
    },
    {
        "city": "Wellington",
        "country": "New Zealand",
        "latitude": -41.2865,
        "longitude": 174.7762
    }
]

In [27]:
import requests

#0 c'est paris
city = cities[0]

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": city["latitude"],
    "longitude": city["longitude"],
    "current": "temperature_2m,wind_speed_10m,weather_code",
    "wind_speed_unit": "ms"
}

response = requests.get(url, params=params, timeout=10)

response.raise_for_status()

print(response.json())

{'latitude': 48.853382, 'longitude': 2.355208, 'generationtime_ms': 0.0756978988647461, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 36.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'wind_speed_10m': 'm/s', 'weather_code': 'wmo code'}, 'current': {'time': '2026-11-10T14:00', 'interval': 900, 'temperature_2m': 12.2, 'wind_speed_10m': 2.05, 'weather_code': 61}}


In [28]:
weather_data = response.json()
weather_data["city"] = city["city"]
weather_data["country"] = city["country"]

producer.send("weather_stream", value=weather_data)
producer.flush()

In [48]:
consumer = KafkaConsumer(
    "weather_stream",
    bootstrap_servers="kafka:9092",
    auto_offset_reset="earliest",
    value_deserializer=lambda message: json.loads(message.decode("utf-8"))
)

for message in consumer:
    print(f"Received message: {message.value}")

Received message: {'msg': 'Hello, Kafka'}
Received message: {'msg': 'Hello, Kafka'}
Received message: {'latitude': 48.853382, 'longitude': 2.355208, 'generationtime_ms': 0.07534027099609375, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 36.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'wind_speed_10m': 'm/s', 'weather_code': 'wmo code'}, 'current': {'time': '2026-11-10T13:45', 'interval': 900, 'temperature_2m': 12.3, 'wind_speed_10m': 1.91, 'weather_code': 80}, 'city': 'Paris', 'country': 'France'}
Received message: {'latitude': 48.853382, 'longitude': 2.355208, 'generationtime_ms': 0.07534027099609375, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 36.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'wind_speed_10m': 'm/s', 'weather_code': 'wmo code'}, 'current': {'time': '2026-11-10T13:45', 'interval': 900, 'temperature_2m

KeyboardInterrupt: 

In [51]:
import time

while True:
    for city in cities:
        params = {
            "latitude": city["latitude"],
            "longitude": city["longitude"],
            "current": "temperature_2m,wind_speed_10m,weather_code",
            "wind_speed_unit": "ms"
        }

        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()

        weather_data = response.json()
        weather_data["city"] = city["city"]
        weather_data["country"] = city["country"]

        producer.send("weather_stream", value=weather_data)
        print(f"city name : {city['city']}")
    producer.flush()
    time.sleep(30)
    

city name : Paris
city name : Seville
city name : Dubai
city name : Reykjavik
city name : Wellington
city name : Paris
city name : Seville
city name : Dubai
city name : Reykjavik
city name : Wellington


KeyboardInterrupt: 